# 4단계: 로컬 베이스 모델(Qwen2.5-Coder-7B) 서빙 + 베이스라인 예측 생성 (Colab)

목표: Colab T4(16GB)에서 vLLM으로 Qwen2.5-Coder-7B-Instruct(AWQ)를 서빙해 Spider dev(1,034개)에 대한 조건 2 예측을 생성한다.

채점(test-suite accuracy)은 `test_suite_database`(4.9GB, 로컬에만 있음)가 필요해 이 노트북에서 하지 않는다 -- 여기서는 예측 jsonl만 만들어 다운로드하고, 로컬에서 `scripts/score_predictions.py`로 채점한다.

**사전 준비**: 런타임 유형을 GPU(T4)로 변경해두었는지 확인.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm
!pip uninstall -y -q torchaudio torchvision
!pip install -q git+https://github.com/MumuKim0212/text2sql-rag-vs-finetune.git

## 스키마 메타데이터 업로드

Spider의 `tables.json`은 HF 데이터셋(`xlangai/spider`)에 포함되어 있지 않다 (question/query/db_id만 parquet로 제공됨). 로컬 저장소의 `data/spider/tables.json` (약 810KB)을 아래 업로드 위젯으로 올려준다.

In [ ]:
from google.colab import files

uploaded = files.upload()  # data/spider/tables.json 선택
assert "tables.json" in uploaded, "tables.json을 업로드해야 함"

In [ ]:
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"

llm = LLM(
    model=MODEL_ID,
    quantization="awq",
    dtype="float16",
    gpu_memory_utilization=0.85,
    max_model_len=4096,
)
print("loaded:", MODEL_ID)

In [ ]:
from rag_text2sql.data import load_schemas, format_schema_prompt
from rag_text2sql.models.cloud import SYSTEM_PROMPT

schemas = load_schemas("tables.json")

smoke_schema_prompt = format_schema_prompt(schemas["concert_singer"])
smoke_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Schema:\n{smoke_schema_prompt}\n\nQuestion: How many singers are there?\nSQL:"},
]

smoke_out = llm.chat([smoke_messages], SamplingParams(temperature=0.0, max_tokens=256))
print(smoke_out[0].outputs[0].text.strip())

## Spider dev 전체 생성

cloud baseline(`scripts/run_cloud_baseline.py`)과 동일하게 32개 단위로 배치 처리하며 매 배치마다 append + flush -- 세션이 끊겨도 이미 쓴 줄은 건너뛰고 재개된다.

In [ ]:
import json
from pathlib import Path

from rag_text2sql.data import load_spider

BATCH_SIZE = 32
OUT_PATH = Path("qwen_dev_predictions.jsonl")

dev = load_spider()["dev"]

done = []
if OUT_PATH.exists():
    with OUT_PATH.open(encoding="utf-8") as f:
        done = [json.loads(line) for line in f if line.strip()]
start = len(done)
print(f"Resuming: {start} examples already done" if start else "Starting fresh")

sampling_params = SamplingParams(temperature=0.0, max_tokens=256)

with OUT_PATH.open("a", encoding="utf-8") as f:
    for batch_start in range(start, len(dev), BATCH_SIZE):
        batch = [dev[i] for i in range(batch_start, min(batch_start + BATCH_SIZE, len(dev)))]
        conversations = [
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        f"Schema:\n{format_schema_prompt(schemas[ex['db_id']])}\n\n"
                        f"Question: {ex['question']}\nSQL:"
                    ),
                },
            ]
            for ex in batch
        ]
        outputs = llm.chat(conversations, sampling_params)
        for ex, out in zip(batch, outputs):
            record = {
                "question": ex["question"],
                "db_id": ex["db_id"],
                "gold_sql": ex["query"],
                "pred_sql": out.outputs[0].text.strip(),
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        print(f"[{min(batch_start + BATCH_SIZE, len(dev))}/{len(dev)}]")

print("done:", OUT_PATH)

## 예측 파일 다운로드

다운로드한 `qwen_dev_predictions.jsonl`을 로컬 저장소의 `data/results/local_base/`에 넣고 아래 명령으로 채점:

```bash
uv run python scripts/score_predictions.py --predictions data/results/local_base/qwen_dev_predictions.jsonl --condition local_base
```

In [ ]:
from google.colab import files

files.download(str(OUT_PATH))